In [ ]:
# ============================================================
# SMART 6-HOUR SUPPLEMENTARY VALIDATION — FINAL SAFE SETUP
# ============================================================

import os
import glob
import warnings
import numpy as np
import pandas as pd

from scipy.stats import chi2_contingency, pearsonr
from sklearn.metrics import confusion_matrix, cohen_kappa_score

warnings.filterwarnings("ignore")

OUTPUT_DIR = "smart6_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SMART6_PATTERN = "smart_6hour_*.csv"
MONTHLY_FILE = "monthly_consumption.csv"
LOCKED_CLUSTER_FILE = "clean_fm_clusters.pkl"

CLUSTER_ORDER = [
    "Peak-Exposed",
    "Typical Household",
    "Heavy Continuous",
    "Daytime-Active"
]

CORE_FEATURES = [
    "evening_peak_ratio",
    "night_baseload_ratio",
    "load_uniformity",
    "load_factor",
    "leisure_usage_ratio",
    "log_daily_kwh"
]

def infer_archetype_name_map(feature_table, label_col="final_cluster"):
    """
    Infer stable archetype names from the locked feature profile.
    This avoids unsafe dependence on arbitrary KMeans numeric cluster IDs.
    """
    required = [
        label_col,
        "evening_peak_ratio",
        "night_baseload_ratio",
        "load_factor",
        "log_daily_kwh"
    ]

    missing = [c for c in required if c not in feature_table.columns]
    if missing:
        raise RuntimeError(
            "Cannot infer archetype names because required columns are missing: "
            + str(missing)
        )

    med = feature_table.groupby(label_col)[
        ["evening_peak_ratio", "night_baseload_ratio", "load_factor", "log_daily_kwh"]
    ].median()

    remaining = set(med.index.tolist())
    mapping = {}

    heavy_id = med["load_factor"].idxmax()
    mapping[heavy_id] = "Heavy Continuous"
    remaining.remove(heavy_id)

    peak_id = med.loc[list(remaining), "evening_peak_ratio"].idxmax()
    mapping[peak_id] = "Peak-Exposed"
    remaining.remove(peak_id)

    daytime_id = med.loc[list(remaining), "night_baseload_ratio"].idxmin()
    mapping[daytime_id] = "Daytime-Active"
    remaining.remove(daytime_id)

    typical_id = list(remaining)[0]
    mapping[typical_id] = "Typical Household"

    return dict(sorted(mapping.items(), key=lambda kv: kv[0]))

CLUSTER_NAMES = {}
CLUSTER_IDS = []
CLUSTER_NAME_TO_ID = {}

GOLDEN_START = pd.Timestamp("2024-05-01")
GOLDEN_END = pd.Timestamp("2024-12-01")  # exclusive upper bound for full May-Nov 2024

def normalise_boolean(value):
    if pd.isna(value):
        return np.nan

    text = str(value).strip().lower()

    true_values = {
        "true", "yes", "y", "1", "export", "exporter",
        "active", "available"
    }

    false_values = {
        "false", "no", "n", "0", "non-export", "non exporter",
        "not active", "inactive", "not available"
    }

    if text in true_values:
        return True

    if text in false_values:
        return False

    num = pd.to_numeric(text, errors="coerce")
    if pd.notna(num):
        return bool(num > 0)

    return np.nan

def last_valid(series):
    s = series.dropna()
    if s.empty:
        return np.nan
    return s.iloc[-1]

def safe_cramers_v(table):
    chi2, _, _, _ = chi2_contingency(table)
    n = table.to_numpy().sum()
    denom = n * (min(table.shape) - 1)
    if denom <= 0:
        return np.nan
    return np.sqrt(chi2 / denom)

# ------------------------------------------------------------
# Load locked smart-meter cluster labels
# ------------------------------------------------------------
clean_fm = pd.read_pickle(LOCKED_CLUSTER_FILE)
clean_fm.index = clean_fm.index.astype(str)

if "final_cluster" not in clean_fm.columns:
    raise RuntimeError("clean_fm_clusters.pkl must contain final_cluster.")

labels_df = clean_fm[["final_cluster"]].reset_index()

if "index" in labels_df.columns:
    labels_df = labels_df.rename(columns={"index": "household_ID"})

labels_df["household_ID"] = labels_df["household_ID"].astype(str)

CLUSTER_NAMES = infer_archetype_name_map(clean_fm, label_col="final_cluster")
CLUSTER_NAME_TO_ID = {v: k for k, v in CLUSTER_NAMES.items()}
CLUSTER_IDS = [CLUSTER_NAME_TO_ID[name] for name in CLUSTER_ORDER]

labels_df["cluster_name"] = labels_df["final_cluster"].map(CLUSTER_NAMES)

if labels_df["cluster_name"].isna().any():
    raise RuntimeError("Some final_cluster values could not be mapped to archetype names.")

print("Dynamic cluster-name map inferred from locked feature medians:")
for cluster_id, cluster_name in CLUSTER_NAMES.items():
    print(f"  {cluster_id} -> {cluster_name}")

print("\nLocked clustered households:", labels_df["household_ID"].nunique())
print("\nLocked cluster distribution:")
print(
    labels_df["cluster_name"]
    .value_counts()
    .reindex(CLUSTER_ORDER)
    .fillna(0)
    .astype(int)
    .to_string()
)

# ------------------------------------------------------------
# Load six-hour files
# ------------------------------------------------------------
smart6_files = sorted(glob.glob(SMART6_PATTERN))

if not smart6_files:
    raise FileNotFoundError("No smart_6hour_*.csv files found.")

parts = []

required_cols = [
    "household_ID",
    "BRANCH",
    "DEPOT",
    "SUB_TYPE",
    "E_BILLING",
    "EXPORT",
    "DATE",
    "TIME",
    "TOTAL_IMPORT (kWh)",
    "TOTAL_EXPORT (kWh)"
]

for file in smart6_files:
    df = pd.read_csv(file, low_memory=False)

    available_cols = [c for c in required_cols if c in df.columns]
    missing_cols = [c for c in required_cols if c not in df.columns]

    if "household_ID" not in available_cols:
        raise RuntimeError(f"household_ID missing in {file}")

    temp = df[available_cols].copy()
    temp["source_file"] = os.path.basename(file)

    temp["household_ID"] = temp["household_ID"].astype(str)

    parts.append(temp)

    print(f"Loaded {file}: rows={len(temp):,}, missing optional cols={missing_cols}")

smart6 = pd.concat(parts, ignore_index=True)

smart6["DATE"] = pd.to_datetime(smart6["DATE"], errors="coerce")

if "TIME" in smart6.columns:
    smart6["timestamp"] = pd.to_datetime(
        smart6["DATE"].astype(str) + " " + smart6["TIME"].astype(str),
        errors="coerce"
    )
else:
    smart6["timestamp"] = smart6["DATE"]

smart6["hour"] = smart6["timestamp"].dt.hour
smart6["month"] = smart6["DATE"].dt.to_period("M").dt.to_timestamp()

smart6["TOTAL_IMPORT_NUMERIC"] = pd.to_numeric(
    smart6["TOTAL_IMPORT (kWh)"].astype(str).str.replace(",", "", regex=False),
    errors="coerce"
)

smart6["TOTAL_EXPORT_NUMERIC"] = pd.to_numeric(
    smart6["TOTAL_EXPORT (kWh)"].astype(str).str.replace(",", "", regex=False),
    errors="coerce"
)

smart6["EXPORT_NORMALISED"] = smart6["EXPORT"].map(normalise_boolean)
smart6["E_BILLING_NORMALISED"] = smart6["E_BILLING"].map(normalise_boolean)

print("\nCombined six-hour shape:", smart6.shape)
print("Unique six-hour households:", smart6["household_ID"].nunique())
print("Date range:", smart6["DATE"].min(), "to", smart6["DATE"].max())

In [ ]:
# ============================================================
# CELL 1 — SIX-HOUR FILE COVERAGE AND TEMPORAL AUDIT
# ============================================================

print("=" * 80)
print("SIX-HOUR FILE COVERAGE AND TEMPORAL AUDIT")
print("=" * 80)

file_audit = (
    smart6
    .groupby("source_file")
    .agg(
        rows=("household_ID", "size"),
        unique_households=("household_ID", "nunique"),
        start_date=("DATE", "min"),
        end_date=("DATE", "max")
    )
    .reset_index()
)

print("\nFile-level audit:")
print(file_audit.to_string(index=False))

smart6_ids = set(smart6["household_ID"].unique())
locked_ids = set(labels_df["household_ID"].unique())

coverage_summary = pd.DataFrame([{
    "total_rows_loaded": len(smart6),
    "smart6_unique_households": len(smart6_ids),
    "locked_cluster_households": len(locked_ids),
    "households_in_both": len(smart6_ids & locked_ids),
    "locked_not_in_smart6": len(locked_ids - smart6_ids),
    "smart6_not_in_locked": len(smart6_ids - locked_ids),
    "locked_coverage_percent": len(smart6_ids & locked_ids) / len(locked_ids) * 100,
    "overall_start_date": smart6["DATE"].min(),
    "overall_end_date": smart6["DATE"].max()
}])

print("\nCoverage summary:")
print(coverage_summary.to_string(index=False))

file_audit.to_csv(
    os.path.join(OUTPUT_DIR, "smart6_file_level_coverage_audit.csv"),
    index=False
)

coverage_summary.to_csv(
    os.path.join(OUTPUT_DIR, "smart6_locked_cluster_coverage_summary.csv"),
    index=False
)

print("\nSaved:")
print("- smart6_outputs/smart6_file_level_coverage_audit.csv")
print("- smart6_outputs/smart6_locked_cluster_coverage_summary.csv")

print("\nSafe interpretation:")
print(
    "The six-hour files are supplementary administrative and validation data. "
    "They cover the locked clustering cohort, but they are not used to train "
    "the clustering model."
)

In [ ]:
# ============================================================
# CELL 2 — ADMINISTRATIVE PROFILE BY LOCKED CLUSTER
# Keeps BRANCH, DEPOT, SUB_TYPE, E_BILLING, EXPORT safely
# ============================================================

print("=" * 80)
print("ADMINISTRATIVE PROFILE BY LOCKED CLUSTER")
print("=" * 80)

# ------------------------------------------------------------
# TEMPORALLY ALIGNED ADMINISTRATIVE METADATA
#
# Resolution priority:
# 1. Latest known value during the May-November 2024
#    clustering window
# 2. Latest chronologically known value
# 3. Deterministic modal value when no timestamped value exists
# ------------------------------------------------------------

admin_source = smart6[
    [
        "household_ID",
        "timestamp",
        "BRANCH",
        "DEPOT",
        "SUB_TYPE"
    ]
].copy()

admin_source["_row_order"] = np.arange(
    len(admin_source)
)


def clean_admin_text(series):
    """
    Standardise administrative text while preserving
    genuinely missing values.
    """

    cleaned = (
        series
        .astype("string")
        .str.strip()
    )

    missing_tokens = [
        "",
        "nan",
        "none",
        "null",
        "<na>"
    ]

    return cleaned.mask(
        cleaned.str.lower().isin(
            missing_tokens
        )
    )


for column in [
    "BRANCH",
    "DEPOT",
    "SUB_TYPE"
]:
    admin_source[column] = clean_admin_text(
        admin_source[column]
    )


def resolve_temporal_admin_value(
    frame,
    value_column
):
    """
    Resolve one administrative value per household.

    Priority:
    1. Latest value during the golden window
    2. Latest chronological value
    3. Deterministic modal fallback
    """

    valid = frame[
        [
            "household_ID",
            "timestamp",
            "_row_order",
            value_column
        ]
    ].dropna(
        subset=[value_column]
    ).copy()

    # --------------------------------------------------------
    # Deterministic modal fallback
    # --------------------------------------------------------

    modal_counts = (
        valid
        .groupby(
            [
                "household_ID",
                value_column
            ],
            dropna=False
        )
        .size()
        .rename("record_count")
        .reset_index()
    )

    modal_value = (
        modal_counts
        .sort_values(
            [
                "household_ID",
                "record_count",
                value_column
            ],
            ascending=[
                True,
                False,
                True
            ]
        )
        .drop_duplicates(
            subset="household_ID",
            keep="first"
        )
        .set_index(
            "household_ID"
        )[value_column]
        .rename(
            value_column
        )
    )

    # --------------------------------------------------------
    # Latest chronologically parsed value
    # --------------------------------------------------------

    dated = (
        valid
        .dropna(
            subset=["timestamp"]
        )
        .sort_values(
            [
                "household_ID",
                "timestamp",
                "_row_order"
            ]
        )
    )

    latest_chronological = (
        dated
        .drop_duplicates(
            subset="household_ID",
            keep="last"
        )
        .set_index(
            "household_ID"
        )[value_column]
        .rename(
            value_column
        )
    )

    # --------------------------------------------------------
    # Latest value during the clustering golden window
    # --------------------------------------------------------

    golden_dated = dated[
        (
            dated["timestamp"]
            >= GOLDEN_START
        )
        & (
            dated["timestamp"]
            < GOLDEN_END
        )
    ]

    latest_golden = (
        golden_dated
        .drop_duplicates(
            subset="household_ID",
            keep="last"
        )
        .set_index(
            "household_ID"
        )[value_column]
        .rename(
            value_column
        )
    )

    # --------------------------------------------------------
    # Apply resolution priority
    # --------------------------------------------------------

    resolved = (
        latest_golden
        .combine_first(
            latest_chronological
        )
        .combine_first(
            modal_value
        )
    )

    resolution_source = pd.Series(
        "modal_fallback",
        index=resolved.index,
        dtype="string"
    )

    chronological_ids = (
        resolution_source.index
        .intersection(
            latest_chronological.index
        )
    )

    resolution_source.loc[
        chronological_ids
    ] = "latest_chronological"

    golden_ids = (
        resolution_source.index
        .intersection(
            latest_golden.index
        )
    )

    resolution_source.loc[
        golden_ids
    ] = "latest_golden_window"

    return pd.DataFrame(
        {
            value_column:
                resolved,
            f"{value_column}_RESOLUTION_SOURCE":
                resolution_source
        }
    )


branch_resolved = resolve_temporal_admin_value(
    admin_source,
    "BRANCH"
)

depot_resolved = resolve_temporal_admin_value(
    admin_source,
    "DEPOT"
)

subtype_resolved = resolve_temporal_admin_value(
    admin_source,
    "SUB_TYPE"
)


# ------------------------------------------------------------
# AUDIT ADMINISTRATIVE VALUE CONFLICTS
# ------------------------------------------------------------

print(
    "\nAdministrative metadata conflict audit:"
)

for column in [
    "BRANCH",
    "DEPOT",
    "SUB_TYPE"
]:

    multiple_value_households = int(
        (
            admin_source
            .groupby(
                "household_ID"
            )[column]
            .nunique(
                dropna=True
            )
            > 1
        ).sum()
    )

    print(
        f"  Households with multiple "
        f"{column} values: "
        f"{multiple_value_households}"
    )


# ------------------------------------------------------------
# E-BILLING AND EXPORT FLAGS
#
# These remain historical "ever observed true" indicators
# across the complete six-hour administrative record.
# ------------------------------------------------------------

historical_flags = (
    smart6
    .groupby(
        "household_ID"
    )
    .agg(
        E_BILLING_FLAG=(
            "E_BILLING_NORMALISED",
            lambda values: (
                bool(
                    values
                    .dropna()
                    .astype(bool)
                    .any()
                )
                if values.notna().any()
                else np.nan
            )
        ),
        ADMIN_EXPORT_FLAG=(
            "EXPORT_NORMALISED",
            lambda values: (
                bool(
                    values
                    .dropna()
                    .astype(bool)
                    .any()
                )
                if values.notna().any()
                else np.nan
            )
        )
    )
)


# ------------------------------------------------------------
# ONE ADMINISTRATIVE RECORD PER HOUSEHOLD
# ------------------------------------------------------------

admin_hh = (
    pd.concat(
        [
            branch_resolved,
            depot_resolved,
            subtype_resolved
        ],
        axis=1
    )
    .join(
        historical_flags,
        how="outer"
    )
    .reset_index()
)

if "index" in admin_hh.columns:
    admin_hh = admin_hh.rename(
        columns={
            "index":
                "household_ID"
        }
    )

admin_hh["household_ID"] = (
    admin_hh["household_ID"]
    .astype(str)
)

assert (
    admin_hh["household_ID"]
    .duplicated()
    .sum()
    == 0
), (
    "Administrative resolution produced "
    "duplicate household IDs."
)


# ------------------------------------------------------------
# SAFE PHASE CLASSIFICATION
# ------------------------------------------------------------

subtype_text = (
    admin_hh["SUB_TYPE"]
    .astype("string")
    .str.lower()
)

single_phase_mask = (
    subtype_text
    .str.contains(
        r"single\s*phase|singlephase|1\s*phase",
        regex=True,
        na=False
    )
)

three_phase_mask = (
    subtype_text
    .str.contains(
        r"3\s*phase|3phase|three\s*phase",
        regex=True,
        na=False
    )
)

admin_hh["PHASE"] = np.select(
    [
        single_phase_mask,
        three_phase_mask
    ],
    [
        "Single Phase",
        "Three Phase"
    ],
    default="Unknown"
)


# ------------------------------------------------------------
# REPORT RESOLUTION SOURCES
# ------------------------------------------------------------

print(
    "\nAdministrative metadata resolution sources:"
)

for column in [
    "BRANCH",
    "DEPOT",
    "SUB_TYPE"
]:

    source_column = (
        f"{column}_RESOLUTION_SOURCE"
    )

    print(
        f"\n{column}:"
    )

    print(
        admin_hh[
            source_column
        ]
        .value_counts(
            dropna=False
        )
        .to_string()
    )

print(
    "\nPhase classification counts:"
)

print(
    admin_hh["PHASE"]
    .value_counts(
        dropna=False
    )
    .to_string()
)

admin_cluster = labels_df.merge(
    admin_hh,
    on="household_ID",
    how="inner"
)

print("Households matched:", len(admin_cluster), "of", len(labels_df))

# ------------------------------------------------------------
# Admin export flag by cluster
# ------------------------------------------------------------
print("\n--- ADMIN EXPORT FLAG BY CLUSTER (%) ---")
export_table = (
    pd.crosstab(
        admin_cluster["cluster_name"],
        admin_cluster["ADMIN_EXPORT_FLAG"],
        normalize="index"
    ) * 100
)

print(export_table.round(1).to_string())

# ------------------------------------------------------------
# Phase type by cluster
# ------------------------------------------------------------
print("\n--- PHASE TYPE BY CLUSTER (%) ---")
phase_table = (
    pd.crosstab(
        admin_cluster["cluster_name"],
        admin_cluster["PHASE"],
        normalize="index"
    ) * 100
)

print(phase_table.round(1).to_string())

# ------------------------------------------------------------
# Branch distribution
# ------------------------------------------------------------
print("\n--- CLUSTER DISTRIBUTION BY CEB BRANCH (%) ---")
branch_table = (
    pd.crosstab(
        admin_cluster["BRANCH"],
        admin_cluster["cluster_name"],
        normalize="index"
    ) * 100
)

branch_counts = pd.crosstab(
    admin_cluster["BRANCH"],
    admin_cluster["cluster_name"]
).sum(axis=1)

branch_table["Total_N"] = branch_counts

print(
    branch_table
    .reindex(columns=CLUSTER_ORDER + ["Total_N"])
    .round(1)
    .sort_values("Peak-Exposed", ascending=False)
    .to_string()
)

# ------------------------------------------------------------
# Depot distribution
# ------------------------------------------------------------
print("\n--- CLUSTER DISTRIBUTION BY DEPOT (%) ---")
depot_table = (
    pd.crosstab(
        admin_cluster["DEPOT"],
        admin_cluster["cluster_name"],
        normalize="index"
    ) * 100
)

depot_counts = pd.crosstab(
    admin_cluster["DEPOT"],
    admin_cluster["cluster_name"]
).sum(axis=1)

depot_table["Total_N"] = depot_counts

print(
    depot_table
    .reindex(columns=CLUSTER_ORDER + ["Total_N"])
    .round(1)
    .sort_values("Peak-Exposed", ascending=False)
    .to_string()
)

# ------------------------------------------------------------
# E-billing imbalance check
# ------------------------------------------------------------
print("\n--- E-BILLING IMBALANCE CHECK ---")

ebill_raw = pd.crosstab(
    admin_cluster["final_cluster"],
    admin_cluster["E_BILLING_FLAG"]
)

chi2, p, dof, expected = chi2_contingency(ebill_raw)
v = safe_cramers_v(ebill_raw)

ebill_pct = (
    pd.crosstab(
        admin_cluster["cluster_name"],
        admin_cluster["E_BILLING_FLAG"],
        normalize="index"
    ) * 100
)

print("\nE-billing percentage by cluster:")
print(ebill_pct.round(1).to_string())

print(f"\nChi-square={chi2:.3f}, df={dof}, p={p:.6g}, Cramer's V={v:.3f}")

admin_cluster.to_csv(
    os.path.join(OUTPUT_DIR, "smart6_household_admin_profile_locked_clusters.csv"),
    index=False
)

export_table.to_csv(
    os.path.join(OUTPUT_DIR, "smart6_admin_export_flag_by_cluster_percent.csv")
)

phase_table.to_csv(
    os.path.join(OUTPUT_DIR, "smart6_phase_type_by_cluster_percent.csv")
)

branch_table.to_csv(
    os.path.join(OUTPUT_DIR, "smart6_branch_distribution_by_cluster_percent.csv")
)

depot_table.to_csv(
    os.path.join(OUTPUT_DIR, "smart6_depot_distribution_by_cluster_percent.csv")
)

ebill_pct.to_csv(
    os.path.join(OUTPUT_DIR, "smart6_ebilling_by_cluster_percent.csv")
)

print("\nSaved administrative outputs.")

print("\nSafe interpretation:")

print(
    "BRANCH, DEPOT and SUB_TYPE were resolved using the latest "
    "available value during the May-November 2024 clustering "
    "window, with chronological and modal fallbacks where needed."
)

print(
    "Phase type is reported only when SUB_TYPE explicitly identifies "
    "a single-phase or three-phase connection. Missing or unrecognized "
    "subtypes remain Unknown."
)

print(
    "Administrative variables are descriptive only. Three-phase share "
    "may indicate higher-capacity connections, but it does not establish "
    "commercial or industrial status."
)

print(
    "E-billing and administrative EXPORT are historical ever-observed "
    "flags in this notebook and must not be interpreted as causal factors."
)

In [ ]:
# ============================================================
# CELL 3 — SIX-HOUR DATA-QUALITY DIAGNOSTIC
# ============================================================

print("=" * 80)
print("SIX-HOUR DATA-QUALITY DIAGNOSTIC")
print("=" * 80)

required_quality_cols = [
    "household_ID",
    "DATE",
    "TIME",
    "TOTAL_IMPORT (kWh)"
]

missing_quality = [
    c for c in required_quality_cols
    if c not in smart6.columns
]

if missing_quality:
    raise RuntimeError(f"Missing required diagnostic columns: {missing_quality}")

print("\nMissing values:")
print(smart6[required_quality_cols].isna().sum().to_string())

print("\nTotal rows loaded:", f"{len(smart6):,}")
print("Unique households:", f"{smart6['household_ID'].nunique():,}")
print("Date range:", smart6["DATE"].min(), "to", smart6["DATE"].max())

# ------------------------------------------------------------
# Time-slot distribution
# ------------------------------------------------------------
print("\nHour distribution:")
hour_counts = smart6["hour"].value_counts().sort_index()
print(hour_counts.to_string())

# ------------------------------------------------------------
# Duplicate household-date-hour rows
# ------------------------------------------------------------
duplicate_hdh = smart6.duplicated(
    subset=["household_ID", "DATE", "hour"]
).sum()

print("\nDuplicate household + date + hour rows:", f"{duplicate_hdh:,}")

# ------------------------------------------------------------
# Household-day slot coverage
# ------------------------------------------------------------
day_slot_counts = (
    smart6
    .groupby(["household_ID", "DATE"])["hour"]
    .nunique()
    .reset_index(name="unique_hour_slots")
)

print("\nDistribution of unique hour slots per household-day:")
print(day_slot_counts["unique_hour_slots"].value_counts().sort_index().to_string())

exactly_4 = (day_slot_counts["unique_hour_slots"] == 4).sum()
total_days = len(day_slot_counts)

print(
    f"\nHousehold-days with exactly 4 unique hour slots: "
    f"{exactly_4:,} / {total_days:,} "
    f"({exactly_4 / total_days * 100:.2f}%)"
)

# ------------------------------------------------------------
# Import register range
# ------------------------------------------------------------
print("\nTOTAL_IMPORT numeric diagnostic:")
print(smart6["TOTAL_IMPORT_NUMERIC"].describe().round(3).to_string())

zero_count = (smart6["TOTAL_IMPORT_NUMERIC"] == 0).sum()
negative_count = (smart6["TOTAL_IMPORT_NUMERIC"] < 0).sum()

print("\nNegative import values:", f"{negative_count:,}")
print("Zero import values:", f"{zero_count:,}")

quality_summary = pd.DataFrame([{
    "total_rows": len(smart6),
    "unique_households": smart6["household_ID"].nunique(),
    "duplicate_household_date_hour_rows": duplicate_hdh,
    "household_days": total_days,
    "household_days_exactly_4_slots": exactly_4,
    "percent_household_days_exactly_4_slots": exactly_4 / total_days * 100,
    "missing_total_import": smart6["TOTAL_IMPORT_NUMERIC"].isna().sum(),
    "negative_total_import": negative_count,
    "zero_total_import": zero_count,
    "min_total_import": smart6["TOTAL_IMPORT_NUMERIC"].min(),
    "max_total_import": smart6["TOTAL_IMPORT_NUMERIC"].max()
}])

quality_summary.to_csv(
    os.path.join(OUTPUT_DIR, "smart6_data_quality_summary.csv"),
    index=False
)

day_slot_counts.to_csv(
    os.path.join(OUTPUT_DIR, "smart6_household_day_hour_slot_counts.csv"),
    index=False
)

print("\nSaved:")
print("- smart6_outputs/smart6_data_quality_summary.csv")
print("- smart6_outputs/smart6_household_day_hour_slot_counts.csv")

print("\nSafe interpretation:")
print(
    "The six-hour data contain cumulative register readings, duplicate "
    "household-date-hour rows, and irregular hour slots. Therefore, raw rows "
    "should not be treated directly as clean six-hour interval consumption."
)

In [ ]:
# ============================================================
# CELL 4 — REGISTER ISOLATION + SIX-HOUR VS MONTHLY BILLING VALIDATION
# Correct method: daily max register, then monthly first-last difference
# ============================================================

print("=" * 80)
print("REGISTER ISOLATION + SIX-HOUR VS MONTHLY BILLING VALIDATION")
print("=" * 80)

# ------------------------------------------------------------
# Isolate primary cumulative register using daily maximum
# This reproduces the validated method from the old notebook.
# ------------------------------------------------------------
raw_import = smart6[
    ["household_ID", "DATE", "TOTAL_IMPORT_NUMERIC"]
].dropna().copy()

raw_import = raw_import.sort_values(["household_ID", "DATE"])

daily_max = (
    raw_import
    .groupby(["household_ID", "DATE"])["TOTAL_IMPORT_NUMERIC"]
    .max()
    .reset_index()
)

daily_max["month"] = daily_max["DATE"].dt.to_period("M").dt.to_timestamp()

monthly_6h = (
    daily_max
    .sort_values(["household_ID", "DATE"])
    .groupby(["household_ID", "month"])["TOTAL_IMPORT_NUMERIC"]
    .agg(["first", "last"])
    .reset_index()
)

monthly_6h["consumption_6h"] = (
    monthly_6h["last"] - monthly_6h["first"]
)

# Remove meter resets and extreme odometer artifacts.
monthly_6h = monthly_6h[
    (monthly_6h["consumption_6h"] >= 0) &
    (monthly_6h["consumption_6h"] < 3000)
].copy()

print("\nDerived monthly consumption from six-hour register:")
print(monthly_6h["consumption_6h"].describe().round(3).to_string())

# ------------------------------------------------------------
# Load monthly billing
# ------------------------------------------------------------
monthly_billing = pd.read_csv(MONTHLY_FILE, low_memory=False)

required_monthly = ["household_ID", "month", "consumption"]
missing_monthly = [
    c for c in required_monthly
    if c not in monthly_billing.columns
]

if missing_monthly:
    raise RuntimeError(f"Missing monthly billing columns: {missing_monthly}")

monthly_billing["household_ID"] = monthly_billing["household_ID"].astype(str)

monthly_billing["consumption_billing"] = pd.to_numeric(
    monthly_billing["consumption"].astype(str).str.replace(",", "", regex=False),
    errors="coerce"
)

monthly_billing["month"] = (
    pd.to_datetime(monthly_billing["month"], errors="coerce")
    .dt.to_period("M")
    .dt.to_timestamp()
)

# ------------------------------------------------------------
# Cross-validate monthly derived six-hour consumption with billing
# ------------------------------------------------------------
cross = pd.merge(
    monthly_6h,
    monthly_billing[["household_ID", "month", "consumption_billing"]],
    on=["household_ID", "month"],
    how="inner"
)

cross = cross.dropna(
    subset=["consumption_6h", "consumption_billing"]
).copy()

corr, pval = pearsonr(
    cross["consumption_6h"],
    cross["consumption_billing"]
)

mae = (
    cross["consumption_6h"] -
    cross["consumption_billing"]
).abs().mean()

bias = (
    cross["consumption_6h"] -
    cross["consumption_billing"]
).mean()

validation_summary = pd.DataFrame([{
    "matched_household_month_pairs": len(cross),
    "matched_households": cross["household_ID"].nunique(),
    "pearson_r": corr,
    "pearson_p": pval,
    "mae_kwh_per_month": mae,
    "mean_bias_6h_minus_billing": bias
}])

print("\nSix-hour vs billing cross-validation:")
print(validation_summary.round(4).to_string(index=False))

# ------------------------------------------------------------
# Correlation by cluster
# ------------------------------------------------------------
cross_cluster = cross.merge(
    labels_df[["household_ID", "final_cluster", "cluster_name"]],
    on="household_ID",
    how="inner"
)

cluster_corr_rows = []

for cluster_id, cluster_name in CLUSTER_NAMES.items():
    sub = cross_cluster[cross_cluster["final_cluster"] == cluster_id]

    if len(sub) > 10:
        r, p_cluster = pearsonr(
            sub["consumption_6h"],
            sub["consumption_billing"]
        )

        cluster_corr_rows.append({
            "cluster": cluster_name,
            "n_pairs": len(sub),
            "pearson_r": r,
            "mae": (
                sub["consumption_6h"] -
                sub["consumption_billing"]
            ).abs().mean()
        })

cluster_corr = pd.DataFrame(cluster_corr_rows)

print("\nCorrelation by cluster:")
print(cluster_corr.round(3).to_string(index=False))

# ------------------------------------------------------------
# Matched vs unmatched monthly-billing coverage check
# ------------------------------------------------------------
billing_ids = set(monthly_billing["household_ID"].dropna().astype(str))
locked_ids = set(labels_df["household_ID"].astype(str))

matched_ids = locked_ids & billing_ids

monthly_6h_smart = monthly_6h.merge(
    labels_df[["household_ID", "final_cluster", "cluster_name"]],
    on="household_ID",
    how="inner"
)

monthly_6h_smart["billing_match_status"] = np.where(
    monthly_6h_smart["household_ID"].isin(matched_ids),
    "Matched",
    "Unmatched"
)

hh_level_full = (
    monthly_6h_smart
    .groupby(["household_ID", "final_cluster", "cluster_name", "billing_match_status"])
    ["consumption_6h"]
    .median()
    .reset_index()
)

full_rows = []

for cluster_id, cluster_name in CLUSTER_NAMES.items():
    sub = hh_level_full[hh_level_full["final_cluster"] == cluster_id]

    m = sub.loc[
        sub["billing_match_status"] == "Matched",
        "consumption_6h"
    ].median()

    u = sub.loc[
        sub["billing_match_status"] == "Unmatched",
        "consumption_6h"
    ].median()

    full_rows.append({
        "cluster": cluster_name,
        "matched_median_kwh": m,
        "unmatched_median_kwh": u,
        "unmatched_minus_matched": u - m
    })

full_match_compare = pd.DataFrame(full_rows)

print("\nMatched vs unmatched median monthly consumption — full six-hour period:")
print(full_match_compare.round(1).to_string(index=False))

# ------------------------------------------------------------
# Golden-window matched vs unmatched check
# ------------------------------------------------------------
monthly_6h_golden = monthly_6h_smart[
    (monthly_6h_smart["month"] >= GOLDEN_START) & (monthly_6h_smart["month"] < GOLDEN_END)
].copy()

hh_level_golden = (
    monthly_6h_golden
    .groupby(["household_ID", "final_cluster", "cluster_name", "billing_match_status"])
    ["consumption_6h"]
    .median()
    .reset_index()
)

golden_rows = []

for cluster_id, cluster_name in CLUSTER_NAMES.items():
    sub = hh_level_golden[hh_level_golden["final_cluster"] == cluster_id]

    m = sub.loc[
        sub["billing_match_status"] == "Matched",
        "consumption_6h"
    ].median()

    u = sub.loc[
        sub["billing_match_status"] == "Unmatched",
        "consumption_6h"
    ].median()

    golden_rows.append({
        "cluster": cluster_name,
        "matched_median_kwh": m,
        "unmatched_median_kwh": u,
        "unmatched_minus_matched": u - m
    })

golden_match_compare = pd.DataFrame(golden_rows)

print("\nMatched vs unmatched median monthly consumption — golden window:")
print(golden_match_compare.round(1).to_string(index=False))

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------
monthly_6h.to_csv(
    os.path.join(OUTPUT_DIR, "smart6_monthly_import_dailymax_method.csv"),
    index=False
)

cross.to_csv(
    os.path.join(OUTPUT_DIR, "smart6_vs_monthly_billing_validation_pairs.csv"),
    index=False
)

validation_summary.to_csv(
    os.path.join(OUTPUT_DIR, "smart6_vs_monthly_billing_validation_summary.csv"),
    index=False
)

cluster_corr.to_csv(
    os.path.join(OUTPUT_DIR, "smart6_vs_monthly_billing_cluster_correlations.csv"),
    index=False
)

full_match_compare.to_csv(
    os.path.join(OUTPUT_DIR, "smart6_billing_matched_unmatched_full_period.csv"),
    index=False
)

golden_match_compare.to_csv(
    os.path.join(OUTPUT_DIR, "smart6_billing_matched_unmatched_golden_window.csv"),
    index=False
)

print("\nSaved smart6 monthly validation outputs.")

print("\nSafe interpretation:")
print(
    "The daily-maximum register method is externally validated against monthly "
    "billing. The strong correlation supports using derived six-hour monthly "
    "import for supplementary validation, but it is not used for clustering. "
    "Matched/unmatched billing comparisons are coverage checks, not causal bias proof."
)


In [ ]:
# ============================================================
# CELL 5 — CORRECTED EXPORT-FLAG AUDIT (PUBLIC DRAFT)
# Final safe export conclusion
# ============================================================

print("=" * 80)
print("CORRECTED SUPPLEMENTARY EXPORT-FLAG AUDIT")
print("=" * 80)

export_hh = (
    smart6
    .groupby("household_ID")
    .agg(
        admin_export_flag=(
            "EXPORT_NORMALISED",
            lambda x: bool(x.dropna().astype(bool).any()) if x.notna().any() else np.nan
        ),
        export_min=("TOTAL_EXPORT_NUMERIC", "min"),
        export_max=("TOTAL_EXPORT_NUMERIC", "max"),
        positive_export_rows=("TOTAL_EXPORT_NUMERIC", lambda x: int((x > 0).sum()))
    )
    .reset_index()
)

export_hh["export_delta_6h"] = (
    export_hh["export_max"] -
    export_hh["export_min"]
)

export_hh["measured_export_above_1kwh"] = (
    export_hh["export_delta_6h"] > 1.0
)

audit = labels_df.merge(
    export_hh,
    on="household_ID",
    how="left"
)

known = audit.dropna(
    subset=["admin_export_flag", "export_delta_6h"]
).copy()

known["admin_export_flag"] = known["admin_export_flag"].astype(bool)
known["measured_export_above_1kwh"] = known["measured_export_above_1kwh"].astype(bool)

cm = confusion_matrix(
    known["measured_export_above_1kwh"],
    known["admin_export_flag"],
    labels=[False, True]
)

tn, fp, fn, tp = cm.ravel()

agreement = (tn + tp) / cm.sum()
kappa = cohen_kappa_score(
    known["measured_export_above_1kwh"],
    known["admin_export_flag"]
)

sensitivity = tp / (tp + fn) if (tp + fn) else np.nan
specificity = tn / (tn + fp) if (tn + fp) else np.nan
positive_predictive_value = tp / (tp + fp) if (tp + fp) else np.nan

cm_df = pd.DataFrame(
    cm,
    index=["Measured No", "Measured Yes"],
    columns=["Admin No", "Admin Yes"]
)

summary = pd.DataFrame([{
    "known_clustered_households_compared": len(known),
    "raw_agreement": agreement,
    "cohens_kappa": kappa,
    "sensitivity": sensitivity,
    "specificity": specificity,
    "positive_predictive_value": positive_predictive_value,
    "true_negative": tn,
    "false_positive": fp,
    "false_negative": fn,
    "true_positive": tp,
    "disagreements": fp + fn
}])

print("Known clustered households compared:", len(known))

print("\nConfusion matrix:")
print(cm_df.to_string())

print("\nCorrected export audit summary:")
print(summary.round(4).to_string(index=False))

disagreements = known[
    known["admin_export_flag"] != known["measured_export_above_1kwh"]
].copy()

print("\nDisagreements:", len(disagreements))

print("Individual disagreement records are intentionally not displayed.")

audit.to_csv(
    os.path.join(OUTPUT_DIR, "smart6_corrected_export_audit_household_level.csv"),
    index=False
)

summary.to_csv(
    os.path.join(OUTPUT_DIR, "smart6_corrected_export_audit_summary.csv"),
    index=False
)

cm_df.to_csv(
    os.path.join(OUTPUT_DIR, "smart6_corrected_export_confusion_matrix.csv")
)

disagreements.to_csv(
    os.path.join(OUTPUT_DIR, "smart6_export_flag_disagreements_corrected.csv"),
    index=False
)

print("\nSaved corrected export audit outputs.")

print("\nSafe interpretation:")
print(
    "Administrative EXPORT and measured six-hour export-register movement are "
    "related but non-equivalent indicators. High raw agreement is mainly caused "
    "by many non-exporting households. Cohen's kappa and positive predictive "
    "value show that positive export agreement is weak. Therefore, this is a "
    "supplementary data-quality audit, not proof that all solar households were removed."
)

# The source notebook also contains a single-household register case audit.
# That case-specific investigation is intentionally excluded from this public draft.


In [ ]:
# ============================================================
# FINAL DECISION AUDIT — SIX-HOUR VALIDATION NOTEBOOK
# ============================================================

print("=" * 80)
print("FINAL DECISION AUDIT — SIX-HOUR VALIDATION NOTEBOOK")
print("=" * 80)

decision_audit = pd.DataFrame([
    {
        "section": "Six-hour file coverage",
        "status": "Final supporting evidence",
        "decision": "Keep",
        "safe_use": f"Shows six-hour files cover all locked {len(labels_df):,} clustering households.",
        "not_safe_use": "Do not claim six-hour data were used to train the clustering model."
    },
    {
        "section": "Administrative profile",
        "status": "Exploratory / descriptive",
        "decision": "Keep",
        "safe_use": "Use BRANCH, DEPOT, SUB_TYPE, E_BILLING, and EXPORT as supplementary admin descriptions.",
        "not_safe_use": "Do not claim commercial/industrial status from three-phase meters."
    },
    {
        "section": "Six-hour data-quality diagnostic",
        "status": "Final limitation evidence",
        "decision": "Keep",
        "safe_use": "Use to justify why raw six-hour rows are not treated as clean interval consumption.",
        "not_safe_use": "Do not directly sum raw six-hour TOTAL_IMPORT rows."
    },
    {
        "section": "Register isolation + monthly billing validation",
        "status": "Final supporting validation",
        "decision": "Keep",
        "safe_use": "Use daily-max register isolation and monthly first-last difference as supplementary validation.",
        "not_safe_use": "Do not use this as part of KMeans clustering training."
    },
    {
        "section": "Matched vs unmatched comparison",
        "status": "Exploratory coverage check",
        "decision": "Keep",
        "safe_use": "Use to discuss possible monthly-billing coverage differences.",
        "not_safe_use": "Do not call it causal selection bias."
    },
    {
        "section": "Corrected export audit",
        "status": "Final export/data-quality audit",
        "decision": "Keep",
        "safe_use": "Report admin EXPORT and measured export-register movement as related but non-equivalent.",
        "not_safe_use": "Do not claim solar filter was triply confirmed."
    }
])

print(decision_audit.to_string(index=False))

decision_audit.to_csv(
    os.path.join(OUTPUT_DIR, "smart6_final_decision_audit.csv"),
    index=False
)

print("\nSaved:")
print("- smart6_outputs/smart6_final_decision_audit.csv")

print("\nFinal notebook decision:")
print(
    "This notebook is used for supplementary six-hour validation and administrative "
    "auditing only. It supports the locked clustering study but does not change the "
    "final clustering model, cohort, or labels."
)